# Social Vulnerability and Health Outcomes in America — Phase 1

**Author:** Mohammad Haider 
**Date:** April 2026 
**Phase:** 1 of 4 — Foundation

---

## The Question

If we know how socioeconomically vulnerable a community is, how well can we predict the health of the people who live there?

Health outcomes in America vary dramatically across geography. A diabetes diagnosis, a mental health crisis, or a chronic disease isn't just a medical event — it often correlates strongly with where a person lives and the conditions of their community.

This analysis joins two CDC datasets at the census tract level — the **Social Vulnerability Index (SVI)** and **PLACES** — to quantify that relationship.

## What This Notebook Does

1. Downloads SVI and PLACES data from CDC (one-time, cached locally)
2. Joins them on census tract FIPS code
3. Computes correlations between vulnerability dimensions and health outcomes
4. Builds a national choropleth showing the geography of the disparity
5. Surfaces state-level patterns

## Caveats Up Front

- This is **observational data**. We can describe associations but not causation.
- Both datasets are estimates with their own uncertainty bounds.
- Census tract is a coarse unit — there's variation within tracts we can't see.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import requests

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 100

# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS = PROJECT_ROOT / 'outputs'

for p in [DATA_RAW, DATA_PROCESSED, OUTPUTS / 'figures']:
    p.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Pandas version: {pd.__version__}')

## Step 1: Download the Data

**SVI 2022** (most recent): A CSV with 16 vulnerability indicators per census tract, plus four summary themes (socioeconomic, household, racial/ethnic minority, housing/transportation) and an overall ranking.

**PLACES 2024 release** (most recent): Estimated prevalence of 36 chronic conditions and behaviors per census tract.

Both datasets are public, no API key needed. We cache locally so we only download once.

**Important:** URLs occasionally change. If a download fails, check:
- SVI: https://www.atsdr.cdc.gov/placeandhealth/svi/data_documentation_download.html
- PLACES: https://www.cdc.gov/places/index.html

In [ ]:
# CDC dataset URLs
# Note: these may need updating if CDC changes their hosting
SVI_URL = 'https://svi.cdc.gov/Documents/Data/2022/csv/SVI_2022_US.csv'
PLACES_URL = 'https://chronicdata.cdc.gov/api/views/cwsq-ngmh/rows.csv?accessType=DOWNLOAD'

def download_if_missing(url: str, dest: Path) -> Path:
    """Download a file if not already present locally."""
    if dest.exists():
        print(f'Already cached: {dest.name} ({dest.stat().st_size / 1e6:.1f} MB)')
        return dest
    print(f'Downloading {url} ...')
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'Saved to {dest} ({dest.stat().st_size / 1e6:.1f} MB)')
    return dest

svi_path = download_if_missing(SVI_URL, DATA_RAW / 'svi_2022.csv')
places_path = download_if_missing(PLACES_URL, DATA_RAW / 'places_2024.csv')

## Step 2: Load and Inspect SVI

In [ ]:
# Load SVI - keep FIPS as string to preserve leading zeros
svi = pd.read_csv(svi_path, dtype={'FIPS': str, 'STCNTY': str})
print(f'SVI shape: {svi.shape}')
svi.head(3)

In [ ]:
# The columns we care about for Phase 1:
# - FIPS: census tract identifier
# - STATE, COUNTY: human-readable location
# - RPL_THEMES: overall percentile ranking (0-1, higher = more vulnerable)
# - RPL_THEME1-4: percentile rankings for each theme
#     Theme 1: Socioeconomic Status
#     Theme 2: Household Characteristics
#     Theme 3: Racial & Ethnic Minority Status
#     Theme 4: Housing Type & Transportation

svi_cols = ['FIPS', 'STATE', 'COUNTY', 'E_TOTPOP',
            'RPL_THEMES', 'RPL_THEME1', 'RPL_THEME2', 'RPL_THEME3', 'RPL_THEME4']

svi_clean = svi[svi_cols].copy()
svi_clean.columns = ['fips', 'state', 'county', 'population',
                     'svi_overall', 'svi_socioeconomic', 'svi_household',
                     'svi_minority', 'svi_housing']

# CDC uses -999 to indicate missing data
for col in ['svi_overall', 'svi_socioeconomic', 'svi_household', 'svi_minority', 'svi_housing']:
    svi_clean[col] = svi_clean[col].replace(-999, np.nan)

print(f'Tracts with valid overall SVI: {svi_clean["svi_overall"].notna().sum():,}')
svi_clean.describe()

## Step 3: Load and Inspect PLACES

PLACES comes in long format — one row per (tract, measure) pair. We'll need to pivot it to wide format for analysis.

In [ ]:
places_raw = pd.read_csv(places_path, dtype={'LocationID': str})
print(f'PLACES shape (long format): {places_raw.shape}')
print(f'Unique measures: {places_raw["MeasureId"].nunique()}')
places_raw[['LocationID', 'MeasureId', 'Measure', 'Data_Value']].head(5)

In [ ]:
# For Phase 1, focus on five flagship outcomes that span chronic disease,
# mental health, and behavioral risk factors
FOCUS_MEASURES = {
    'DIABETES': 'Diabetes',
    'MHLTH': 'Poor Mental Health',
    'CHD': 'Coronary Heart Disease',
    'OBESITY': 'Obesity',
    'CSMOKING': 'Current Smoking',
    'BPHIGH': 'High Blood Pressure'
}

# Pivot to wide: one row per tract, one column per measure
places = (places_raw[places_raw['MeasureId'].isin(FOCUS_MEASURES.keys())]
          .pivot_table(index='LocationID',
                       columns='MeasureId',
                       values='Data_Value',
                       aggfunc='first')
          .reset_index()
          .rename(columns={'LocationID': 'fips'}))

print(f'PLACES wide shape: {places.shape}')
places.head(3)

## Step 4: Join the Datasets

Both datasets use 11-digit FIPS codes. We do an inner join to keep only tracts present in both.

In [ ]:
df = svi_clean.merge(places, on='fips', how='inner')
print(f'Joined dataset: {df.shape[0]:,} census tracts')
print(f'Population covered: {df["population"].sum():,}')
df.head(3)

In [ ]:
# Save the joined dataset for reuse in later phases
df.to_parquet(DATA_PROCESSED / 'svi_places_joined.parquet')
print(f'Saved to {DATA_PROCESSED / "svi_places_joined.parquet"}')

## Step 5: The Headline Finding — Correlations

How strongly does each SVI theme correlate with each health outcome? 
Spoiler: very strongly, but the strength varies by condition.

In [ ]:
svi_cols_list = ['svi_overall', 'svi_socioeconomic', 'svi_household', 'svi_minority', 'svi_housing']
outcome_cols = list(FOCUS_MEASURES.keys())

corr = df[svi_cols_list + outcome_cols].corr().loc[svi_cols_list, outcome_cols]

# Pretty labels
corr.index = ['Overall SVI', 'Socioeconomic', 'Household', 'Minority Status', 'Housing/Transport']
corr.columns = [FOCUS_MEASURES[c] for c in corr.columns]

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, cbar_kws={'label': 'Pearson correlation'},
            ax=ax, linewidths=0.5)
ax.set_title('Social Vulnerability vs. Health Outcomes\n(Pearson correlation across ~73,000 US census tracts)',
             fontsize=13, pad=15)
ax.set_ylabel('SVI Component')
ax.set_xlabel('Health Outcome')
plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### What this shows

**TODO after running:** Write 2-3 sentences interpreting the heatmap. Look for:
- Which SVI theme correlates most strongly with each outcome?
- Are there outcomes that DON'T track with vulnerability? (Often interesting to flag.)
- Any sign-flipping (negative correlations) that needs explanation?

## Step 6: The Headline Scatter Plot

One image, one finding. Pick the highest-correlation pair and show it cleanly.

In [ ]:
# Diabetes vs. SVI - typically the strongest signal
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(df['svi_overall'], df['DIABETES'],
           alpha=0.08, s=8, color='#1f4e79', edgecolors='none')

# Trend line via rolling median (more robust than regression on this scale)
tmp = df[['svi_overall', 'DIABETES']].dropna().sort_values('svi_overall')
tmp['rolling_median'] = tmp['DIABETES'].rolling(2000, center=True, min_periods=500).median()
ax.plot(tmp['svi_overall'], tmp['rolling_median'],
        color='#c8102e', linewidth=3, label='Rolling median')

ax.set_xlabel('Social Vulnerability Index (percentile)', fontsize=12)
ax.set_ylabel('Diabetes Prevalence (%)', fontsize=12)
ax.set_title('More vulnerable communities have substantially higher diabetes rates',
             fontsize=14, pad=15)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Annotate the headline number
low_svi_diabetes = df[df['svi_overall'] < 0.2]['DIABETES'].median()
high_svi_diabetes = df[df['svi_overall'] > 0.8]['DIABETES'].median()
ax.axhline(low_svi_diabetes, color='gray', linestyle='--', alpha=0.4)
ax.axhline(high_svi_diabetes, color='gray', linestyle='--', alpha=0.4)
ax.text(0.02, low_svi_diabetes + 0.3, f'Low-vulnerability tracts: {low_svi_diabetes:.1f}%',
        fontsize=10, color='gray')
ax.text(0.98, high_svi_diabetes - 0.5, f'High-vulnerability tracts: {high_svi_diabetes:.1f}%',
        fontsize=10, color='gray', ha='right')

plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'diabetes_vs_svi.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDiabetes prevalence in low-vulnerability tracts: {low_svi_diabetes:.1f}%')
print(f'Diabetes prevalence in high-vulnerability tracts: {high_svi_diabetes:.1f}%')
print(f'Ratio: {high_svi_diabetes / low_svi_diabetes:.2f}x')

## Step 7: State-Level Disparities

National averages obscure regional patterns. Where is the SVI-health relationship sharpest?

In [ ]:
# For each state, compute the within-state correlation between SVI and diabetes
state_corr = (df.dropna(subset=['svi_overall', 'DIABETES'])
                .groupby('state')
                .apply(lambda g: g['svi_overall'].corr(g['DIABETES']))
                .sort_values(ascending=True)
                .rename('correlation')
                .reset_index())

fig, ax = plt.subplots(figsize=(9, 12))
colors = ['#c8102e' if c > 0.7 else '#1f4e79' if c > 0.5 else '#888' for c in state_corr['correlation']]
ax.barh(state_corr['state'], state_corr['correlation'], color=colors)
ax.set_xlabel('Within-state correlation: SVI vs. Diabetes prevalence', fontsize=11)
ax.set_title('States where vulnerability most strongly predicts diabetes',
             fontsize=13, pad=15)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax.axvline(0.7, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS / 'figures' / 'state_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 1 Summary

**TODO after running:** Write 4-6 sentences summarizing what we learned.

Suggested structure:
1. The headline number (e.g., "diabetes rates are X.Xx higher in the most vulnerable tracts")
2. Which vulnerability dimension matters most
3. Where the disparity is sharpest geographically
4. The explicit caveat about correlation vs. causation
5. What Phase 2 will explore

## Next Steps (Phase 2)

- [ ] Build interactive choropleth map with Folium/Plotly
- [ ] Explore "twin tracts" — pairs with similar SVI but divergent outcomes
- [ ] Add urban/rural dimension (CDC's NCHS Urban-Rural Classification)
- [ ] Sensitivity analysis: do findings hold when we control for population age?